In [1]:
import os

os.environ["RUN_PART2"] = "1"
os.environ["TRAIN_STEPS"] = "1000"
os.environ["GEN_STEPS"] = "30"
os.environ["DATASET_NAME"] = "lambdalabs/naruto-blip-captions"

In [2]:
import subprocess,sys,random,numpy as np,torch
SEED=266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
commit=subprocess.run(['git','rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip()
print({'commit':commit,'python':sys.version.split()[0],'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu','seed':SEED})

{'commit': '', 'python': '3.13.2', 'torch': '2.15.0+cu132', 'gpu': 'NVIDIA GeForce RTX 4090', 'seed': 266}


# Part 2 — Fine-tuning Diffusion Models
Set `DATASET_NAME`, `BASE_MODEL`, and `RUN_PART2=1` before running all cells. The code performs two same-budget LoRA runs (rank 4 and rank 16), saves adapters, and generates the required base/run-1/run-2 images. Metrics must be computed by the evaluation cells after the real run.

In [3]:
import os, json, sys
from pathlib import Path
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'part2'/'src').exists()),Path.cwd())
sys.path.insert(0,str(ROOT/'part2'/'src'))
DATASET_NAME=os.environ.get('DATASET_NAME','lambdalabs/naruto-blip-captions')
BASE_MODEL=os.environ.get('BASE_MODEL','stable-diffusion-v1-5/stable-diffusion-v1-5')
OUT_DIR=ROOT/'part2'/'outputs'
from train_and_eval import prepare_dataset, train_lora, generate_images
print({'dataset':DATASET_NAME,'base_model':BASE_MODEL,'output':str(OUT_DIR),'run_part2':os.environ.get('RUN_PART2','0')})

{'dataset': 'lambdalabs/naruto-blip-captions', 'base_model': 'stable-diffusion-v1-5/stable-diffusion-v1-5', 'output': 'C:\\Users\\saketh1\\Desktop\\lab2_solution\\part2\\outputs', 'run_part2': '1'}


In [4]:
if os.environ.get('RUN_PART2','0') == '1':
    metadata=prepare_dataset(DATASET_NAME,OUT_DIR/'data')
    print({k:metadata[k] for k in ['original_count','valid_count','removed_corrupt','removed_duplicates','train_count','test_count']})
else:
    print('Preparation/training is disabled. Set RUN_PART2=1 on the GPU lab machine; no result was fabricated.')

Repo card metadata block was not found. Setting CardData to empty.


{'original_count': 1221, 'valid_count': 1221, 'removed_corrupt': 0, 'removed_duplicates': 0, 'train_count': 1098, 'test_count': 123}


In [5]:
if os.environ.get('RUN_PART2','0') == '1':
    in_domain=[row['caption'] for row in metadata['test'][:10]]
    prompts=in_domain + ['a realistic red panda reading a newspaper in a city park','a watercolor lighthouse during a thunderstorm','a robot chef preparing pasta in a small kitchen','a clay sculpture of a mountain village','a cinematic underwater research station','two characters, one in red and one in blue, standing side by side','three characters sharing one umbrella in the rain','a character holding a lantern beside a sleeping dragon','two characters racing bicycles through a colorful market','a small character and a giant character comparing maps']
    assert len(prompts)==20
    (OUT_DIR/'prompts_before_training.json').write_text(json.dumps(prompts,indent=2))
    print('committed prompts before training:',len(prompts))

committed prompts before training: 20


In [6]:
import train_and_eval
print(train_and_eval.__file__)

C:\Users\saketh1\Desktop\lab2_solution\part2\src\train_and_eval.py


In [7]:
if os.environ.get('RUN_PART2','0') == '1':
    run1=train_lora(BASE_MODEL,DATASET_NAME,OUT_DIR,'lora_rank4',rank=4,steps=int(os.environ.get('TRAIN_STEPS','1000')),batch_size=1,learning_rate=1e-4)
    run2=train_lora(BASE_MODEL,DATASET_NAME,OUT_DIR,'lora_rank16',rank=16,steps=int(os.environ.get('TRAIN_STEPS','1000')),batch_size=1,learning_rate=1e-4)
    print(run1); print(run2)

Repo card metadata block was not found. Setting CardData to empty.
Repo card metadata block was not found. Setting CardData to empty.


TrainingRecord(run='lora_rank4', rank=4, learning_rate=0.0001, steps=1000, batch_size=1, trainable_parameters=797184, training_seconds=194.61542869987898, gpu_hours=0.05405984130552194, peak_gpu_memory_gb=6.641356468200684, adapter_path='C:\\Users\\saketh1\\Desktop\\lab2_solution\\part2\\outputs\\lora_rank4')
TrainingRecord(run='lora_rank16', rank=16, learning_rate=0.0001, steps=1000, batch_size=1, trainable_parameters=3188736, training_seconds=191.59739849995822, gpu_hours=0.05322149958332173, peak_gpu_memory_gb=10.677676677703857, adapter_path='C:\\Users\\saketh1\\Desktop\\lab2_solution\\part2\\outputs\\lora_rank16')


In [8]:
if os.environ.get('RUN_PART2','0') == '1':
    prompts=json.loads((OUT_DIR/'prompts_before_training.json').read_text())
    manifest=generate_images(BASE_MODEL,{'run1':OUT_DIR/'lora_rank4','run2':OUT_DIR/'lora_rank16'},prompts,OUT_DIR/'generated',seeds=(0,1),steps=int(os.environ.get('GEN_STEPS','30')))
    print('generated',len(manifest),'images')

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.
Potential NSFW content was detected in one or more images. A black image will be returned instead. Try again with a different prompt and/or seed.


generated 120 images
